<a href="https://colab.research.google.com/github/JoaoNogaroli/mestrado/blob/main/matriz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y -qq glpk-utils

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pyomo.environ import *

# ─────────────────────────────────────────
# PARÂMETROS
# ─────────────────────────────────────────
ROWS, COLS   = 300, 300
PROB_PAREDE  = 0.40   # ~28% de paredes
PROB_ARMADILHA = 0.10 # ~7% de armadilhas
CUSTO_LIVRE  = 1
CUSTO_ARMADILHA = 10
SEED = None  # None = aleatório a cada execução; coloque um inteiro para fixar

# ─────────────────────────────────────────
# 1. GERAR MATRIZ ALEATÓRIA
# ─────────────────────────────────────────
rng = np.random.default_rng(SEED)

def gerar_grid(rows, cols):
    """
    Gera grid aleatório garantindo que origem e destino
    sempre sejam acessíveis e que exista pelo menos um caminho.
    Valores: 0=parede, 1=livre, 10=armadilha
    """
    while True:
        grid = np.ones((rows, cols), dtype=int)

        # Gera paredes aleatórias
        paredes = rng.random((rows, cols)) < PROB_PAREDE
        grid[paredes] = 0

        # Gera armadilhas nas células livres
        livres = (grid == 1)
        armadilhas = rng.random((rows, cols)) < PROB_ARMADILHA
        grid[livres & armadilhas] = 10

        # Garante origem e destino livres
        grid[0, 0] = 1
        grid[rows-1, cols-1] = 1

        # Verifica conectividade via BFS
        if bfs_conectado(grid, rows, cols):
            return grid

def bfs_conectado(grid, rows, cols):
    """BFS para verificar se existe caminho de (0,0) até (R-1,C-1)."""
    visited = np.zeros((rows, cols), dtype=bool)
    queue = [(0, 0)]
    visited[0, 0] = True
    dirs = [(-1,0),(1,0),(0,-1),(0,1)]
    while queue:
        r, c = queue.pop(0)
        if r == rows-1 and c == cols-1:
            return True
        for dr, dc in dirs:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr][nc] and grid[nr][nc] != 0:
                visited[nr][nc] = True
                queue.append((nr, nc))
    return False

print(f"Gerando labirinto {ROWS, COLS}...")
grid = gerar_grid(ROWS, COLS)

n_paredes = np.sum(grid == 0)
n_armadilhas = np.sum(grid == 10)
n_livres = np.sum(grid == 1)
print(f"  Células livres: {n_livres} | Armadilhas: {n_armadilhas} | Paredes: {n_paredes}")

# ─────────────────────────────────────────
# 2. CONSTRUIR GRAFO (nós e arcos)
# ─────────────────────────────────────────
origem  = (0, 0)
destino = (ROWS-1, COLS-1)
dirs    = [(-1,0),(1,0),(0,-1),(0,1)]

nos   = [(r, c) for r in range(ROWS) for c in range(COLS) if grid[r][c] != 0]
arcos = []
custo = {}

for (r, c) in nos:
    for dr, dc in dirs:
        nr, nc = r+dr, c+dc
        if (nr, nc) in set(nos):
            arcos.append(((r,c),(nr,nc)))
            custo[((r,c),(nr,nc))] = int(grid[nr][nc])

nos_set = set(nos)
print(f"  Nós no grafo: {len(nos)} | Arcos: {len(arcos)}")

# ─────────────────────────────────────────
# 3. MODELO PYOMO — CAMINHO MÍNIMO
# ─────────────────────────────────────────
print("\nConstruindo modelo Pyomo...")
model = ConcreteModel()

# Variáveis de decisão: x[i,j] = 1 se arco (i,j) está no caminho
model.x = Var(arcos, domain=Binary)

# Função objetivo: minimizar custo total do caminho
model.obj = Objective(
    expr=sum(custo[a] * model.x[a] for a in arcos),
    sense=minimize
)

# Restrições de fluxo em cada nó
arcos_set = set(arcos)

def fluxo_rule(model, r, c):
    no = (r, c)
    vizinhos_saida  = [(no, (r+dr, c+dc)) for dr, dc in dirs
                       if (r+dr, c+dc) in nos_set and (no, (r+dr, c+dc)) in arcos_set]
    vizinhos_entrada = [((r+dr, c+dc), no) for dr, dc in dirs
                        if (r+dr, c+dc) in nos_set and ((r+dr, c+dc), no) in arcos_set]

    if not vizinhos_saida and not vizinhos_entrada:
        return Constraint.Feasible

    saida  = sum(model.x[a] for a in vizinhos_saida)
    entrada = sum(model.x[a] for a in vizinhos_entrada)

    if no == origem:
        return saida - entrada == 1
    elif no == destino:
        return saida - entrada == -1
    else:
        return saida - entrada == 0

model.fluxo = Constraint(nos, rule=fluxo_rule)

# ─────────────────────────────────────────
# 4. RESOLVER
# ─────────────────────────────────────────
print("Resolvendo com GLPK...")
solver = SolverFactory('glpk', executable='/usr/bin/glpsol')
resultado = solver.solve(model, tee=False)

print(f"  Status: {resultado.solver.termination_condition}")
custo_total = int(value(model.obj))
print(f"  Custo total do caminho ótimo: {custo_total}")

# Extrair caminho
caminho_arcos = [(a[0], a[1]) for a in arcos if value(model.x[a]) > 0.5]

# Reconstruir sequência de células
def reconstruir_caminho(arcos_solucao, origem, destino):
    grafo = {}
    for (u, v) in arcos_solucao:
        grafo[u] = v
    path = [origem]
    cur = origem
    while cur != destino:
        cur = grafo[cur]
        path.append(cur)
    return path

caminho = reconstruir_caminho(caminho_arcos, origem, destino)
n_armadilhas_caminho = sum(1 for (r,c) in caminho if grid[r][c] == 10)
print(f"  Células no caminho: {len(caminho)}")
print(f"  Armadilhas atravessadas: {n_armadilhas_caminho}")

# ─────────────────────────────────────────
# 5. VISUALIZAÇÃO
# ─────────────────────────────────────────
print("\nGerando visualização...")

# Monta matriz de visualização
viz = grid.copy().astype(float)
# 0=parede, 1=livre, 2=armadilha(grid==10→2), 3=caminho, 4=origem, 5=destino
viz[viz == 10] = 2
for (r, c) in caminho:
    viz[r, c] = 3
viz[0, 0] = 4
viz[ROWS-1, COLS-1] = 5

cores = ['#2C2C2A',  # 0 - parede (cinza escuro)
         '#E1F5EE',  # 1 - livre (verde claro)
         '#FCEBEB',  # 2 - armadilha (vermelho claro)
         '#378ADD',  # 3 - caminho ótimo (azul)
         '#EF9F27',  # 4 - origem (laranja)
         '#1D9E75']  # 5 - destino (verde)

cmap = ListedColormap(cores)

fig, axes = plt.subplots(1, 2, figsize=(16, 8),
                          gridspec_kw={'width_ratios': [3, 1]})

# ── Labirinto ──
ax = axes[0]
ax.imshow(viz, cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Labirinto 50×50 — Caminho Mínimo (Shortest Path)',
             fontsize=14, fontweight='bold', pad=12)

# Marcadores de origem e destino
ax.text(0, 0, '🐭', ha='center', va='center', fontsize=10)
ax.text(COLS-1, ROWS-1, '🚪', ha='center', va='center', fontsize=10)

# Legenda
patches = [
    mpatches.Patch(color='#2C2C2A', label='Parede (custo ∞)'),
    mpatches.Patch(color='#E1F5EE', label='Livre (custo 1)'),
    mpatches.Patch(color='#FCEBEB', label=f'Armadilha (custo {CUSTO_ARMADILHA})'),
    mpatches.Patch(color='#378ADD', label='Caminho ótimo'),
    mpatches.Patch(color='#EF9F27', label='Origem / Destino'),
]
ax.legend(handles=patches, loc='upper right', fontsize=9,
          framealpha=0.9, edgecolor='#ccc')

# ── Painel de resultados ──
ax2 = axes[1]
ax2.axis('off')

info = [
    ('Dimensão', f'{ROWS} × {COLS}'),
    ('', ''),
    ('Nós no grafo', f'{len(nos)}'),
    ('Arcos no grafo', f'{len(arcos)}'),
    ('', ''),
    ('Células livres', f'{n_livres}'),
    ('Armadilhas', f'{n_armadilhas}'),
    ('Paredes', f'{n_paredes}'),
    ('', ''),
    ('── SOLUÇÃO ──', ''),
    ('Custo total', f'{custo_total}'),
    ('Células no caminho', f'{len(caminho)}'),
    ('Armadilhas no caminho', f'{n_armadilhas_caminho}'),
    ('', ''),
    ('── MODELO ──', ''),
    ('Variáveis binárias', f'{len(arcos)}'),
    ('Restrições de fluxo', f'{len(nos)}'),
    ('Solver', 'GLPK'),
]

y = 0.97
for label, val in info:
    if label.startswith('──'):
        ax2.text(0.05, y, label, fontsize=10, fontweight='bold',
                 color='#444', transform=ax2.transAxes, va='top')
    elif label == '':
        pass
    else:
        ax2.text(0.05, y, label + ':', fontsize=9, color='#666',
                 transform=ax2.transAxes, va='top')
        ax2.text(0.95, y, val, fontsize=9, fontweight='bold',
                 color='#222', transform=ax2.transAxes, va='top', ha='right')
    y -= 0.052

plt.tight_layout()
plt.savefig(f'labirinto_shortest_path_{ROWS}_{COLS}.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Imagem salva!")

Gerando labirinto (300, 300)...
  Células livres: 48552 | Armadilhas: 5382 | Paredes: 36066
  Nós no grafo: 53934 | Arcos: 129124

Construindo modelo Pyomo...
Resolvendo com GLPK...
  Status: optimal
  Custo total do caminho ótimo: 1228
  Células no caminho: 869
  Armadilhas atravessadas: 40

Gerando visualização...


/tmp/ipykernel_33406/1160953564.py:253: UserWarning: Glyph 128682 (\N{DOOR}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_33406/1160953564.py:254: UserWarning: Glyph 128682 (\N{DOOR}) missing from font(s) DejaVu Sans.
  plt.savefig(f'labirinto_shortest_path_{ROWS}_{COLS}.png',


Imagem salva!


## APRENDENDO OQ FIZ ALI EM CIMA

In [199]:
import numpy as np

ROWS, COLS = 7, 7

In [201]:
from collections import deque

def tem_caminho(grid, ROWS, COLS):
    origem  = (0, 0)
    destino = (ROWS-1, COLS-1)

    visitado = set()
    fila = deque([origem])
    visitado.add(origem)

    dirs = [(-1,0),(1,0),(0,-1),(0,1)]

    while fila:
        r, c = fila.popleft()


        if (r, c) == destino:
            return True  # achou caminho!

        print("============================")
        print("Verificação da posição NO GRID: ", (r,c))
        print("Valor NO GRID: ", grid[r][c])
        print("===============")
        for dr, dc in dirs:
            nr, nc = r+dr, c+dc
            if dr == -1:
              print("Verificando a direção PRA CIMA: ", (dr,dc))
            if dr == 1:
              print("Verificando a direção PRA BAIXO: ", (dr,dc))
            if dc == -1:
              print("Verificando a direção PRA ESQUERDA: ", (dr,dc))
            if dc == 1:
              print("Verificando a direção PRA DIREITA: ", (dr,dc))
            try:
              print(f"Resultado da verificação: Posição {nr,nc},Valor no Grid: {grid[nr,nc]}----- \n-----------\n")
            except Exception as e:
              continue
            if (0 <= nr < ROWS and 0 <= nc < COLS
                and (nr,nc) not in visitado
                and grid[nr][nc] != 0):
                visitado.add((nr,nc))
                fila.append((nr,nc))
    print("===============")
    print("GRID NAO PODE SER USADO")
    print("===============")
    return False  # não achou


while True:
    # gera matriz aleatória
    grid = np.ones((ROWS, COLS), dtype=int)
    for r in range(ROWS):
        for c in range(COLS):
            if np.random.random() < 0.80:
                grid[r][c] = 0
    grid[0][0] = 1
    grid[ROWS-1][COLS-1] = 1
    print('=============')
    print('NOVO GRID')
    print('=============')
    print(grid)
    # só sai do loop se tiver caminho
    if tem_caminho(grid, ROWS, COLS):
        break

print("Labirinto válido gerado!")
print(grid)

A saída de streaming foi truncada nas últimas 5000 linhas.
GRID NAO PODE SER USADO
NOVO GRID
[[1 1 0 1 0 0 0]
 [0 1 0 0 0 0 0]
 [0 1 0 1 0 0 0]
 [0 0 1 0 0 0 0]
 [0 0 1 0 0 0 1]
 [0 0 0 1 0 0 0]
 [1 1 0 0 1 1 1]]
Verificação da posição NO GRID:  (0, 0)
Valor NO GRID:  1
Verificando a direção PRA CIMA:  (-1, 0)
Resultado da verificação: Posição (-1, 0),Valor no Grid: 1----- 
-----------

Verificando a direção PRA BAIXO:  (1, 0)
Resultado da verificação: Posição (1, 0),Valor no Grid: 0----- 
-----------

Verificando a direção PRA ESQUERDA:  (0, -1)
Resultado da verificação: Posição (0, -1),Valor no Grid: 0----- 
-----------

Verificando a direção PRA DIREITA:  (0, 1)
Resultado da verificação: Posição (0, 1),Valor no Grid: 1----- 
-----------

Verificação da posição NO GRID:  (0, 1)
Valor NO GRID:  1
Verificando a direção PRA CIMA:  (-1, 0)
Resultado da verificação: Posição (-1, 1),Valor no Grid: 1----- 
-----------

Verificando a direção PRA BAIXO:  (1, 0)
Resultado da verificação: Posiç